In [4]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model, load_model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import os

# ========== مسیرها ==========
base_dir = '/content/drive/MyDrive/FPF_PROJECT/'
device_dir = base_dir + 'Revision/Device_Analsys/'
model_path = base_dir + 'trained_classifier_model_whole_partial_finetune.h5'
test_csv_path = device_dir + 'TEST.csv'
base_images = base_dir + 'Whole'

# ========== 1. بارگذاری TEST.csv ==========
test_set = pd.read_csv(test_csv_path)
print(f"تعداد تست: {len(test_set)}")

# ========== 2. ساخت مسیر تصاویر ==========
def fix_image_path(row):
    if 'path' in row and pd.notna(row['path']):
        return os.path.join(base_images, str(row['path']).split('/')[-1])
    elif 'image_path' in row and pd.notna(row['image_path']):
        return os.path.join(base_images, str(row['image_path']).split('/')[-1])
    elif 'file_name' in row and pd.notna(row['file_name']):
        return os.path.join(base_images, str(row['file_name']))
    return None

test_set['img_path'] = test_set.apply(fix_image_path, axis=1)
test_set['img_exists'] = test_set['img_path'].apply(lambda x: os.path.exists(x) if x else False)
print(f"✅ تصاویر موجود: {test_set['img_exists'].sum()}")

# ========== 3. بارگذاری مدل‌ها ==========
classifier = load_model(model_path)

try:
    feature_extractor = Model(inputs=classifier.input,
                              outputs=classifier.get_layer('global_average_pooling2d').output)
except ValueError:
    try:
        feature_extractor = Model(inputs=classifier.input,
                                  outputs=classifier.get_layer('global_average_pooling2d_1').output)
    except ValueError:
        feature_extractor = Model(inputs=classifier.input,
                                  outputs=classifier.get_layer('global_average_pooling2d_2').output)

regression_models = {}
for class_idx in range(4):
    with open(base_dir + f'regressor_class_{class_idx}_whole.pkl', 'rb') as f:
        regression_models[class_idx] = pickle.load(f)

print("✅ همه مدل‌ها بارگذاری شدند")

def preprocess_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0
    return image

# ========== 4. پیش‌بینی ==========
test_valid = test_set[test_set['img_exists']].copy()

# تصاویر خام
print("🔄 بارگذاری تصاویر...")
X_images = np.vstack([tf.stack([preprocess_image(p) for p in test_valid['img_path'].values[i:i+16]]).numpy()
                       for i in range(0, len(test_valid), 16)])

# ویژگی‌ها
print("🔄 استخراج ویژگی...")
X_features = feature_extractor.predict(X_images, verbose=0)

# پیش‌بینی کلاس
print("🔄 پیش‌بینی کلاس...")
class_probs = classifier.predict(X_images, verbose=0)
predicted_classes = np.argmax(class_probs, axis=1)

# پیش‌بینی FPF
print("🔄 پیش‌بینی FPF...")
fpf_predictions = [
    regression_models[pc].predict(feat.reshape(1, -1))[0] if pc in regression_models else np.nan
    for feat, pc in zip(X_features, predicted_classes)
]

test_valid['predicted_class'] = predicted_classes
test_valid['predicted_fpf'] = fpf_predictions
test_valid['abs_error'] = np.abs(test_valid['fpf'] - test_valid['predicted_fpf'])

# ========== 5. گزارش ==========
print("\n" + "="*70)
print("📊 نتایج")
print("="*70)
print(f"MAE کل: {test_valid['abs_error'].mean():.4f}")
print(f"RMSE کل: {np.sqrt((test_valid['abs_error']**2).mean()):.4f}")
print(f"R² کل: {r2_score(test_valid['fpf'], test_valid['predicted_fpf']):.4f}")

# ذخیره
test_valid.to_csv(device_dir + 'test_predictions_full.csv', index=False)
print(f"✅ ذخیره شد.")

تعداد تست: 151
✅ تصاویر موجود: 151


✅ همه مدل‌ها بارگذاری شدند
🔄 بارگذاری تصاویر...
🔄 استخراج ویژگی...
🔄 پیش‌بینی کلاس...


🔄 پیش‌بینی FPF...

📊 نتایج
MAE کل: 1.8471
RMSE کل: 2.2447
R² کل: 0.9896
✅ ذخیره شد.


In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

base_dir = '/content/drive/MyDrive/FPF_PROJECT/'
device_dir = base_dir + 'Revision/Device_Analsys/'

# بارگذاری پیش‌بینی‌ها
test = pd.read_csv(device_dir + 'test_predictions_full.csv')
metadata = pd.read_csv(base_dir + 'source_reviewer_metadata.csv')

# ========== اتصال source_id ==========
def extract_source_id(file_name):
    file_name = str(file_name).upper()
    patterns = {
        'TH-2012-01': 'SRC_0001', 'TH-2013-01': 'SRC_0002',
        'TH-2015-01': 'SRC_0003', 'TH-2015-02': 'SRC_0004',
        'TH-2016-01': 'SRC_0005', 'TH-2017-01': 'SRC_0006',
        'TH-2018-01': 'SRC_0007', 'TH-2018-02': 'SRC_0008',
        'TH-2020-01': 'SRC_0009', 'TH-2020-02': 'SRC_0087',
        'TH-2020-03': 'SRC_0088', 'TH-2020-04': 'SRC_0089',
        'TH-2020-05': 'SRC_0113', 'TH-2020-06': 'SRC_0090',
        'TH-2021-01': 'SRC_0010', 'TH-2021-02': 'SRC_0103',
        'TH-2021-03': 'SRC_0092', 'TH-2022-01': 'SRC_0011',
        'TH-2022-02': 'SRC_0012', 'TH-2022-03': 'SRC_0115',
        'TH-2022-04': 'SRC_0094', 'TH-2023-01': 'SRC_0013',
        'TH-2023-02': 'SRC_0014', 'TH-2024-01': 'SRC_0015',
        'TH-2024-02': 'SRC_0016', 'TH-2025-01': 'SRC_0017',
        'TH-2025-02': 'SRC_0018',
        'TH04_': 'SRC_0004', 'TH06_': 'SRC_0006', 'TH07_': 'SRC_0007',
        'AR1-2017-1': 'SRC_0072', 'AR2-2017-2': 'SRC_0071',
        'AR1-2019': 'SRC_0059', 'AR2-2018': 'SRC_0060',
        'AR-2020-01': 'SRC_0022', 'AR-2020-02': 'SRC_0020',
        'AR-2020-03': 'SRC_0103', 'AR-2020-05': 'SRC_0021',
        'AR-2020-06': 'SRC_0022', 'AR-2020-07': 'SRC_0023',
        'AR-2021-01': 'SRC_0062', 'AR-2021-02': 'SRC_0070',
        'AR-2021-03': 'SRC_0076', 'AR-2021-05': 'SRC_0026',
        'AR-2021-06': 'SRC_0027', 'AR-2021-07': 'SRC_0075',
        'AR-2022-01': 'SRC_0028', 'AR-2022-02': 'SRC_0068',
        'AR-2022-03': 'SRC_0078', 'AR-2022-04': 'SRC_0079',
        'AR-2022-05': 'SRC_0071', 'AR-2022-06': 'SRC_0065',
        'AR-2022-07': 'SRC_0029', 'AR-2022-08': 'SRC_0030',
        'AR-2022-09': 'SRC_0069', 'AR-2022-10': 'SRC_0066',
        'AR-2022-11': 'SRC_0031', 'AR-2022-12': 'SRC_0032',
        'AR-2022-13': 'SRC_0033',
        'AR-2023-01': 'SRC_0034', 'AR-2023-02': 'SRC_0035',
        'AR-2023-04': 'SRC_0104', 'AR-2023-05': 'SRC_0105',
        'AR-2023-06': 'SRC_0036', 'AR-2023-07': 'SRC_0037',
        'AR-2023-08': 'SRC_0038', 'AR-2023-11': 'SRC_0077',
        'AR-2023-12': 'SRC_0039',
        'AR-2024-01': 'SRC_0040', 'AR-2024-02': 'SRC_0106',
        'AR-2024-03': 'SRC_0041', 'AR-2024-04': 'SRC_0042',
        'AR-2024-05': 'SRC_0043', 'AR-2024-06': 'SRC_0044',
        'AR-2024-07': 'SRC_0045', 'AR-2024-08': 'SRC_0046',
        'AR-2024-09': 'SRC_0047', 'AR-2024-11': 'SRC_0048',
        'AR-2025-01': 'SRC_0049', 'AR-2025-02': 'SRC_0107',
        'AR-2025-04': 'SRC_0050', 'AR-2025-05': 'SRC_0051',
        'AR-2025-06': 'SRC_0052', 'AR-2025-07': 'SRC_0067',
        'AR-2025-08': 'SRC_0053', 'AR-2025-09': 'SRC_0054',
        'AR-2025-10': 'SRC_0055', 'AR-2025-11': 'SRC_0056',
        'AR-2025-12': 'SRC_0057', 'AR-2025-13': 'SRC_0058',
        'AR-2025-14': 'SRC_0080', 'AR-2025-15': 'SRC_0081',
        'SIN_AR1-2019': 'SRC_0059', 'SIN_AR7-2019': 'SRC_0061',
        'SIN_AR4-2019': 'SRC_0073',
    }
    for prefix, sid in patterns.items():
        if file_name.startswith(prefix):
            return sid
    return 'UNKNOWN'

test['source_id'] = test['file_name'].apply(extract_source_id)
test = test.merge(
    metadata[['source_id', 'airflow_rates_l_min', 'impactor_types']],
    on='source_id',
    how='left'
)

# ========== تحلیل Airflow ==========
def categorize_airflow(airflow):
    if pd.isna(airflow):
        return '1. Unclear'
    s = str(airflow).strip()
    if s in ['60', '60.0']:
        return '2. 60 (استاندارد)'
    elif any(sep in s for sep in [';', '|', ',']):
        return '3. چندگانه'
    else:
        return '4. غیراستاندارد'

test['airflow_category'] = test['airflow_rates_l_min'].apply(categorize_airflow)

print("="*70)
print("📊 MAE بر اساس Airflow")
print("="*70)
airflow_stats = test.groupby('airflow_category').agg(
    N=('abs_error', 'count'),
    MAE=('abs_error', 'mean'),
    RMSE=('abs_error', lambda x: np.sqrt((x**2).mean())),
    R2=('fpf', lambda x: r2_score(x, test.loc[x.index, 'predicted_fpf']) if len(x) > 1 else np.nan)
).round(4)
print(airflow_stats.to_string())

# ========== تحلیل Device ==========
print("\n" + "="*70)
print("📊 MAE بر اساس Device (فقط N≥5)")
print("="*70)
device_stats = test.groupby('impactor_types').agg(
    N=('abs_error', 'count'),
    MAE=('abs_error', 'mean'),
).round(4)
device_stats = device_stats[device_stats['N'] >= 5].sort_values('N', ascending=False)
print(device_stats.to_string())

# ========== ذخیره ==========
test.to_csv(device_dir + 'test_with_airflow_device.csv', index=False)
print(f"\n✅ فایل 'test_with_airflow_device.csv' ذخیره شد.")

📊 MAE بر اساس Airflow
                    N     MAE    RMSE      R2
airflow_category                             
1. Unclear         22  2.0144  2.4541  0.9780
2. 60 (استاندارد)  64  1.7411  2.1024  0.9913
3. چندگانه          8  1.8388  2.0653  0.9828
4. غیراستاندارد    57  1.9027  2.3370  0.9877

📊 MAE بر اساس Device (فقط N≥5)
                                                               N     MAE
impactor_types                                                          
Twin Stage Impinger (TSI)                                     34  1.6321
Next Generation Impactor (NGI)                                27  1.6055
Next Generation Impactor                                      10  1.0772
Twin-Stage Impinger (TSI)                                     10  2.2561
Fast Screening Impactor (FSI)                                  6  2.2303
Twin Stage Impinger (TSI; British Pharmacopoeia Apparatus A)   6  0.8146
Twin Stage Impinger (TSI; Copley Scientific)                   5  1.1816

✅ فایل 'test

In [6]:
import pandas as pd

base_dir = '/content/drive/MyDrive/FPF_PROJECT/'
device_dir = base_dir + 'Revision/Device_Analsys/'

# بارگذاری داده تحلیل‌شده
test = pd.read_csv(device_dir + 'test_with_airflow_device.csv')

print("="*70)
print("📊 بررسی کامل بودن داده‌های Device")
print("="*70)
print(f"تعداد کل تست: {len(test)}")

# تعداد NaN در impactor_types
nan_devices = test['impactor_types'].isna().sum()
valid_devices = test['impactor_types'].notna().sum()

print(f"\n✅ دارای impactor_types: {valid_devices}")
print(f"❌ بدون impactor_types (NaN): {nan_devices}")

# نمایش نمونه‌هایی که impactor_types ندارند
print("\n" + "="*70)
print("📋 نمونه تصاویری که impactor_types ندارند:")
print("="*70)

missing = test[test['impactor_types'].isna()][['file_name', 'source_id', 'fpf']]
print(missing.head(20).to_string())

# سورس‌های بدون impactor_types
print("\n" + "="*70)
print("📋 سورس‌های بدون impactor_types در تست:")
print("="*70)
missing_sources = missing['source_id'].value_counts()
print(missing_sources.to_string())

# چک: آیا این سورس‌ها در metadata هم impactor_types ندارند؟
metadata = pd.read_csv(base_dir + 'source_reviewer_metadata.csv')

print("\n" + "="*70)
print("📋 چک metadata این سورس‌ها:")
print("="*70)
for sid in missing_sources.index:
    row = metadata[metadata['source_id'] == sid]
    if len(row) > 0:
        device = row['impactor_types'].values[0]
        print(f"{sid}: impactor_types = '{device}'")

📊 بررسی کامل بودن داده‌های Device
تعداد کل تست: 151

✅ دارای impactor_types: 129
❌ بدون impactor_types (NaN): 22

📋 نمونه تصاویری که impactor_types ندارند:
                                   file_name source_id     fpf
0                         AR8-2019-F2-01.png   UNKNOWN  54.270
13                           AR3-2019-F3.png   UNKNOWN  23.714
18                        AR7-2019-F2-01.png   UNKNOWN  60.800
43                        TH1-2019-F6-01.png   UNKNOWN  37.000
52                           AR5-2019-F4.png   UNKNOWN  55.800
53                        AR8-2019-F2-02.png   UNKNOWN  54.270
56                        AR4-2019-F3-10.png   UNKNOWN  64.100
58                        AR4-2019-F3-02.png   UNKNOWN  64.100
67                  Ar-2020-08-T11-25-02.png   UNKNOWN  67.500
68                      TH2-2017-2-F2-02.png   UNKNOWN  31.550
69                      TH2-2017-2-F3-01.png   UNKNOWN  30.400
72                        TH1-2019-F9-03.png   UNKNOWN  38.100
89                      T

In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

base_dir = '/content/drive/MyDrive/FPF_PROJECT/'
device_dir = base_dir + 'Revision/Device_Analsys/'

# بارگذاری پیش‌بینی‌ها
test = pd.read_csv(device_dir + 'test_predictions_full.csv')
metadata = pd.read_csv(base_dir + 'source_reviewer_metadata.csv')

# ========== تابع کامل extract_source_id ==========
def extract_source_id(file_name):
    file_name = str(file_name).upper()

    # همه الگوها (کامل)
    patterns = {
        # OMI_TH
        'OMI_TH-2012-01': 'SRC_0001', 'OMI_TH-2013-01': 'SRC_0002',
        'OMI_TH-2015-01': 'SRC_0003', 'OMI_TH-2015-02': 'SRC_0004',
        'OMI_TH-2016-01': 'SRC_0005', 'OMI_TH-2017-01': 'SRC_0006',
        'OMI_TH-2018-01': 'SRC_0007', 'OMI_TH-2018-02': 'SRC_0008',
        'OMI_TH-2020-01': 'SRC_0009', 'OMI_TH-2021-01': 'SRC_0010',
        'OMI_TH-2022-01': 'SRC_0011', 'OMI_TH-2022-02': 'SRC_0012',
        'OMI_TH-2023-01': 'SRC_0013', 'OMI_TH-2023-02': 'SRC_0014',
        'OMI_TH-2024-01': 'SRC_0015', 'OMI_TH-2024-02': 'SRC_0016',
        'OMI_TH-2025-01': 'SRC_0017', 'OMI_TH-2025-02': 'SRC_0018',
        # TH-20XX
        'TH-2012-01': 'SRC_0001', 'TH-2013-01': 'SRC_0002',
        'TH-2015-01': 'SRC_0003', 'TH-2015-02': 'SRC_0004',
        'TH-2016-01': 'SRC_0005', 'TH-2017-01': 'SRC_0006',
        'TH-2018-01': 'SRC_0007', 'TH-2018-02': 'SRC_0008',
        'TH-2020-01': 'SRC_0009', 'TH-2020-02': 'SRC_0087',
        'TH-2020-03': 'SRC_0088', 'TH-2020-04': 'SRC_0089',
        'TH-2020-05': 'SRC_0113', 'TH-2020-06': 'SRC_0090',
        'TH-2021-01': 'SRC_0010', 'TH-2021-02': 'SRC_0103',
        'TH-2021-03': 'SRC_0092', 'TH-2022-01': 'SRC_0011',
        'TH-2022-02': 'SRC_0012', 'TH-2022-03': 'SRC_0115',
        'TH-2022-04': 'SRC_0094', 'TH-2023-01': 'SRC_0013',
        'TH-2023-02': 'SRC_0014', 'TH-2024-01': 'SRC_0015',
        'TH-2024-02': 'SRC_0016', 'TH-2025-01': 'SRC_0017',
        'TH-2025-02': 'SRC_0018',
        # کوتاه TH
        'TH04_': 'SRC_0004', 'TH06_': 'SRC_0006', 'TH07_': 'SRC_0007',
        'TH1-2019': 'SRC_0011',
        'TH2-2017-2': 'SRC_0002',
        'TH3-2017-3': 'SRC_0003',
        'TH3_2017-3': 'SRC_0003',
        # AR قدیمی
        'AR1-2017-1': 'SRC_0072', 'AR2-2017-2': 'SRC_0071',
        'AR1-2019': 'SRC_0059', 'AR2-2018': 'SRC_0060',
        'AR2-2019': 'SRC_0063', 'AR3-2019': 'SRC_0102',
        'AR4-2019': 'SRC_0073', 'AR5-2019': 'SRC_0061',
        'AR6-2019': 'SRC_0057', 'AR7-2019': 'SRC_0061',
        'AR8-2019': 'SRC_0019',
        # AR-20XX
        'AR-2020-01': 'SRC_0022', 'AR-2020-02': 'SRC_0020',
        'AR-2020-03': 'SRC_0103', 'AR-2020-04': 'SRC_0019',
        'AR-2020-05': 'SRC_0021', 'AR-2020-06': 'SRC_0022',
        'AR-2020-07': 'SRC_0023', 'AR-2020-08': 'SRC_0024',
        'AR-2020-09': 'SRC_0025',
        'AR-2021-01': 'SRC_0062', 'AR-2021-02': 'SRC_0070',
        'AR-2021-03': 'SRC_0076', 'AR-2021-05': 'SRC_0026',
        'AR-2021-06': 'SRC_0027', 'AR-2021-07': 'SRC_0075',
        'AR-2022-01': 'SRC_0028', 'AR-2022-02': 'SRC_0068',
        'AR-2022-03': 'SRC_0078', 'AR-2022-04': 'SRC_0079',
        'AR-2022-05': 'SRC_0071', 'AR-2022-06': 'SRC_0065',
        'AR-2022-07': 'SRC_0029', 'AR-2022-08': 'SRC_0030',
        'AR-2022-09': 'SRC_0069', 'AR-2022-10': 'SRC_0066',
        'AR-2022-11': 'SRC_0031', 'AR-2022-12': 'SRC_0032',
        'AR-2022-13': 'SRC_0033', 'AR-2022-14': 'SRC_0064',
        'AR-2023-01': 'SRC_0034', 'AR-2023-02': 'SRC_0035',
        'AR-2023-04': 'SRC_0104', 'AR-2023-05': 'SRC_0105',
        'AR-2023-06': 'SRC_0036', 'AR-2023-07': 'SRC_0037',
        'AR-2023-08': 'SRC_0038', 'AR-2023-11': 'SRC_0077',
        'AR-2023-12': 'SRC_0039',
        'AR-2024-01': 'SRC_0040', 'AR-2024-02': 'SRC_0106',
        'AR-2024-03': 'SRC_0041', 'AR-2024-04': 'SRC_0042',
        'AR-2024-05': 'SRC_0043', 'AR-2024-06': 'SRC_0044',
        'AR-2024-07': 'SRC_0045', 'AR-2024-08': 'SRC_0046',
        'AR-2024-09': 'SRC_0047', 'AR-2024-11': 'SRC_0048',
        'AR-2025-01': 'SRC_0049', 'AR-2025-02': 'SRC_0107',
        'AR-2025-04': 'SRC_0050', 'AR-2025-05': 'SRC_0051',
        'AR-2025-06': 'SRC_0052', 'AR-2025-07': 'SRC_0067',
        'AR-2025-08': 'SRC_0053', 'AR-2025-09': 'SRC_0054',
        'AR-2025-10': 'SRC_0055', 'AR-2025-11': 'SRC_0056',
        'AR-2025-12': 'SRC_0057', 'AR-2025-13': 'SRC_0058',
        'AR-2025-14': 'SRC_0080', 'AR-2025-15': 'SRC_0081',
        # Ar-2020-08 (حرف بزرگ و کوچک)
        'AR-2020-08': 'SRC_0024',
        # SIN
        'SIN_AR1-2019': 'SRC_0059', 'SIN_AR7-2019': 'SRC_0061',
        'SIN_AR4-2019': 'SRC_0073',
        'SIN_TH04': 'SRC_0004', 'SIN_TH06': 'SRC_0006', 'SIN_TH07': 'SRC_0007',
        'SIN_TH1-2019': 'SRC_0011', 'SIN_TH2-2017-2': 'SRC_0002',
        'SIN_TH3-2017-3': 'SRC_0003', 'SIN_TH3_2017-3': 'SRC_0003',
        'SIN_AR1-2017-1': 'SRC_0072', 'SIN_AR2-2017-2': 'SRC_0071',
        'SIN_AR2-2018': 'SRC_0060', 'SIN_AR2-2019': 'SRC_0063',
        'SIN_AR3-2019': 'SRC_0102', 'SIN_AR4-2019': 'SRC_0073',
        'SIN_AR5-2019': 'SRC_0061', 'SIN_AR6-2019': 'SRC_0057',
        'SIN_AR7-2019': 'SRC_0061', 'SIN_AR8-2019': 'SRC_0019',
        # SET
        'SET_AR-2020-01': 'SRC_0019', 'SET_AR-2020-02': 'SRC_0020',
        'SET_AR-2020-03': 'SRC_0103', 'SET_AR-2020-04': 'SRC_0019',
        'SET_AR-2020-05': 'SRC_0021', 'SET_AR-2020-06': 'SRC_0022',
        'SET_AR-2020-07': 'SRC_0023', 'SET_AR-2020-08': 'SRC_0024',
        'SET_AR-2020-09': 'SRC_0025',
        'SET_AR-2021-01': 'SRC_0062', 'SET_AR-2021-02': 'SRC_0070',
        'SET_AR-2021-03': 'SRC_0076', 'SET_AR-2021-05': 'SRC_0026',
        'SET_AR-2021-06': 'SRC_0027', 'SET_AR-2021-07': 'SRC_0075',
        'SET_AR-2022-01': 'SRC_0028', 'SET_AR-2022-02': 'SRC_0068',
        'SET_AR-2022-03': 'SRC_0078', 'SET_AR-2022-04': 'SRC_0079',
        'SET_AR-2022-05': 'SRC_0071', 'SET_AR-2022-06': 'SRC_0065',
        'SET_AR-2022-07': 'SRC_0029', 'SET_AR-2022-08': 'SRC_0030',
        'SET_AR-2022-09': 'SRC_0069', 'SET_AR-2022-10': 'SRC_0066',
        'SET_AR-2022-11': 'SRC_0031', 'SET_AR-2022-12': 'SRC_0032',
        'SET_AR-2022-13': 'SRC_0033', 'SET_AR-2022-14': 'SRC_0064',
        'SET_AR-2023-01': 'SRC_0034', 'SET_AR-2023-02': 'SRC_0035',
        'SET_AR-2023-04': 'SRC_0104', 'SET_AR-2023-05': 'SRC_0105',
        'SET_AR-2023-06': 'SRC_0036', 'SET_AR-2023-07': 'SRC_0037',
        'SET_AR-2023-08': 'SRC_0038', 'SET_AR-2023-11': 'SRC_0077',
        'SET_AR-2023-12': 'SRC_0039',
        'SET_AR-2024-01': 'SRC_0040', 'SET_AR-2024-02': 'SRC_0106',
        'SET_AR-2024-03': 'SRC_0041', 'SET_AR-2024-04': 'SRC_0042',
        'SET_AR-2024-05': 'SRC_0043', 'SET_AR-2024-06': 'SRC_0044',
        'SET_AR-2024-07': 'SRC_0045', 'SET_AR-2024-08': 'SRC_0046',
        'SET_AR-2024-09': 'SRC_0047', 'SET_AR-2024-11': 'SRC_0048',
        'SET_AR-2025-01': 'SRC_0049', 'SET_AR-2025-02': 'SRC_0107',
        'SET_AR-2025-04': 'SRC_0050', 'SET_AR-2025-05': 'SRC_0051',
        'SET_AR-2025-06': 'SRC_0052', 'SET_AR-2025-07': 'SRC_0067',
        'SET_AR-2025-08': 'SRC_0053', 'SET_AR-2025-09': 'SRC_0054',
        'SET_AR-2025-10': 'SRC_0055', 'SET_AR-2025-11': 'SRC_0056',
        'SET_AR-2025-12': 'SRC_0057', 'SET_AR-2025-13': 'SRC_0058',
        'SET_AR-2025-14': 'SRC_0080', 'SET_AR-2025-15': 'SRC_0081',
        'SET_TH-2020-01': 'SRC_0009', 'SET_TH-2020-02': 'SRC_0087',
        'SET_TH-2020-03': 'SRC_0088', 'SET_TH-2020-04': 'SRC_0089',
        'SET_TH-2020-05': 'SRC_0113', 'SET_TH-2020-06': 'SRC_0090',
        'SET_TH-2021-02': 'SRC_0103', 'SET_TH-2021-03': 'SRC_0092',
        'SET_TH-2022-01': 'SRC_0093', 'SET_TH-2022-02': 'SRC_0114',
        'SET_TH-2022-03': 'SRC_0115', 'SET_TH-2022-04': 'SRC_0094',
        'SET_TH-2023-01': 'SRC_0095', 'SET_TH-2024-01': 'SRC_0096',
        'SET_TH-2025-01': 'SRC_0097',
        # Ar-2020-08 با حرف کوچک
        'AR-2020-08': 'SRC_0024',
    }

    # چک کردن الگوها (به ترتیب طول - طولانی‌ترها اول)
    for prefix, sid in sorted(patterns.items(), key=lambda x: -len(x[0])):
        if file_name.startswith(prefix):
            return sid

    return 'UNKNOWN'

# اعمال تابع
test['source_id'] = test['file_name'].apply(extract_source_id)

# چک تعداد UNKNOWN
unknown_count = (test['source_id'] == 'UNKNOWN').sum()
print(f"تعداد UNKNOWN بعد از اصلاح: {unknown_count}")

if unknown_count > 0:
    print("\nنمونه‌های UNKNOWN:")
    print(test[test['source_id'] == 'UNKNOWN']['file_name'].head(20).to_string())

# اتصال metadata
test = test.merge(
    metadata[['source_id', 'airflow_rates_l_min', 'impactor_types']],
    on='source_id',
    how='left'
)

# ========== تحلیل Airflow ==========
def categorize_airflow(airflow):
    if pd.isna(airflow):
        return '1. Unclear'
    s = str(airflow).strip()
    if s in ['60', '60.0']:
        return '2. 60 (استاندارد)'
    elif any(sep in s for sep in [';', '|', ',']):
        return '3. چندگانه'
    else:
        return '4. غیراستاندارد'

test['airflow_category'] = test['airflow_rates_l_min'].apply(categorize_airflow)

print("\n" + "="*70)
print("📊 MAE بر اساس Airflow")
print("="*70)
airflow_stats = test.groupby('airflow_category').agg(
    N=('abs_error', 'count'),
    MAE=('abs_error', 'mean'),
    RMSE=('abs_error', lambda x: np.sqrt((x**2).mean())),
    R2=('fpf', lambda x: r2_score(x, test.loc[x.index, 'predicted_fpf']) if len(x) > 1 else np.nan)
).round(4)
print(airflow_stats.to_string())

# ========== تحلیل Device ==========
print("\n" + "="*70)
print("📊 MAE بر اساس Device (فقط N≥5)")
print("="*70)
device_stats = test.groupby('impactor_types').agg(
    N=('abs_error', 'count'),
    MAE=('abs_error', 'mean'),
).round(4)
device_stats = device_stats[device_stats['N'] >= 5].sort_values('N', ascending=False)
print(device_stats.to_string())

# ========== تعداد کل ==========
print("\n" + "="*70)
print(f"📊 تعداد کل تست با impactor_types معتبر: {test['impactor_types'].notna().sum()}")
print("="*70)

# ذخیره
test.to_csv(device_dir + 'test_with_airflow_device_v2.csv', index=False)
print(f"\n✅ فایل 'test_with_airflow_device_v2.csv' ذخیره شد.")


تعداد UNKNOWN بعد از اصلاح: 0

📊 MAE بر اساس Airflow
                    N     MAE    RMSE      R2
airflow_category                             
2. 60 (استاندارد)  72  1.6930  2.0391  0.9915
3. چندگانه         12  1.7442  1.9848  0.9919
4. غیراستاندارد    67  2.0312  2.4862  0.9861

📊 MAE بر اساس Device (فقط N≥5)
                                                               N     MAE
impactor_types                                                          
Twin Stage Impinger (TSI)                                     39  1.6324
Next Generation Impactor (NGI)                                34  1.8105
Next Generation Impactor                                      10  1.0772
Twin-Stage Impinger (TSI)                                     10  2.2561
Fast Screening Impactor (FSI)                                  6  2.2303
Twin Stage Impinger (TSI; British Pharmacopoeia Apparatus A)   6  0.8146
Twin Stage Impinger (TSI; Copley Scientific)                   5  1.1816

📊 تعداد کل تست با impactor_